**Step 1- Install & Import**

In [1]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

**Step 2-Dataset**

In [2]:
# Simple sentence dataset
texts_positive = [
    "I love this product",
    "this is amazing",
    "absolutely wonderful experience",
    "best thing ever",
    "i really enjoyed this",
    "fantastic product",
    "very happy with this",
    "this works perfectly",
    "excellent quality",
    "superb experience",
    "highly recommend this",
    "worth every penny",
    "this is great",
    "awesome product",
    "really satisfied",
    "this is very good",
    "great value for money",
    "i like this a lot",
    "very impressive",
    "perfect purchase",
    "this made my day",
    "so happy with this",
    "brilliant product",
    "loved it",
    "this is fantastic",
    "great experience overall",
    "very useful product",
    "top quality",
    "extremely good",
    "nice and smooth",
    "this is excellent",
    "wonderful item",
    "i am impressed",
    "works like a charm",
    "best purchase ever",
    "high quality product",
    "i absolutely love this",
    "this is outstanding",
    "great performance",
    "amazing results",
    "really great",
    "so good",
    "very nice",
    "pleasant experience",
    "loved the quality",
    "happy customer",
    "totally worth it",
    "five stars",
    "super happy",
    "i enjoyed using this"
]

texts_negative = [
    "this is terrible",
    "worst experience ever",
    "i hate this product",
    "absolutely disgusting",
    "never buying this again",
    "not good at all",
    "very disappointing",
    "this is bad",
    "poor quality",
    "waste of money",
    "not worth it",
    "completely useless",
    "i regret buying this",
    "very bad experience",
    "this is awful",
    "horrible product",
    "extremely disappointing",
    "does not work",
    "very poor performance",
    "worst purchase",
    "i dislike this",
    "not satisfied",
    "bad quality",
    "this broke quickly",
    "very unhappy",
    "terrible service",
    "low quality",
    "really bad",
    "i am not happy",
    "waste product",
    "not recommended",
    "this is worst",
    "pathetic product",
    "not useful",
    "cheap quality",
    "very frustrating",
    "disappointed with this",
    "this failed",
    "hated it",
    "very poor",
    "doesn't work properly",
    "not good",
    "awful experience",
    "bad purchase",
    "completely disappointed",
    "useless product",
    "never again",
    "very bad",
    "this is horrible",
    "i hate it",
    "i really hate this product",
    "this is extremely terrible",
    "worst thing i ever bought",
    "i am very disappointed",
    "completely useless item",
    "this is not good at all",
    "very very bad experience",
    "i regret buying this",
    "absolutely horrible product",
    "this is the worst ever"
]

texts = texts_positive + texts_negative
labels = [1]*len(texts_positive) + [0]*len(texts_negative)

**Step 3- Preprocessing & Vocabulary**

In [3]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.split()

# Build Vocabulary
all_words = []
for t in texts:
    all_words.extend(clean(t))

# Count word frequencies
word_counts = Counter(all_words)
max_vocab_size=1000                                            # Limted vocabulary size to avoide noice
special_tokens = ["<PAD>", "<UNK>", "<SOS>", "<EOS>"] 
most_common_words = [w for w, _ in word_counts.most_common(max_vocab_size)]

  # Remove duplicates
most_common_words = [w for w in most_common_words if w not in special_tokens]

vocab = special_tokens + most_common_words

# Word -> index
word2idx = {w: i for i, w in enumerate(vocab)}

#index -> word
idx2word = {i: w for w, i in word2idx.items()}

print("Vocabulary size:", len(vocab))
print("Sample word2idx:", dict(list(word2idx.items())[:5]))
print("Sample idx2word:", dict(list(idx2word.items())[:6]))

Vocabulary size: 113
Sample word2idx: {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3, 'this': 4}
Sample idx2word: {0: '<PAD>', 1: '<UNK>', 2: '<SOS>', 3: '<EOS>', 4: 'this', 5: 'i'}


**Step 4-Encode & Pad**

In [4]:
def encode_and_pad(text, word2idx, max_len=10):
    tokens = clean(text)

    # Unknown words
    encoded = [word2idx.get(w, word2idx["<UNK>"]) for w in tokens] 

    # Special tokens
    encoded = [word2idx["<SOS>"]] + encoded + [word2idx["<EOS>"]]

    if len(encoded) < max_len:
        encoded += [word2idx["<PAD>"]] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]

    return encoded

# ________________Test____________________
sample = encode_and_pad("I love this product", word2idx)
print("Encoded:", sample)

Encoded: [2, 5, 34, 4, 8, 3, 0, 0, 0, 0]


**Step 5- Pytorch Dataset**

In [5]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, word2idx, max_len=10):
        self.x = [encode_and_pad(t, word2idx, max_len) for t in texts]
        self.y = labels
    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return(
            torch.tensor(self.x[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

# Create Dataset and Dataloader
dataset = SentimentDataset(texts, labels, word2idx)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print("Dataset size:", len(dataset))
print("Batches  ", len(dataloader))

Dataset size: 110
Batches   28


 **Step 6 — LSTM Sentiment Model**

In [6]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0) 
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _ , (hidden, _) = self.lstm(embedded)
        hidden = self.dropout(hidden.squeeze(0))
        out = self.fc(hidden)

        return out.squeeze(1)

# Initialize model
model = SentimentLSTM(
    vocab_size = len(vocab),
    embed_size = 32,
    hidden_size = 64
)
print(model)

SentimentLSTM(
  (embedding): Embedding(113, 32, padding_idx=0)
  (lstm): LSTM(32, 64, batch_first=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=64, out_features=1, bias=True)
)


**Step 7 — Train**

In [7]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 40

for epoch in range(epochs):
    model.train()
    total_loss = 0

    # 🔹 Training
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()

        predictions = model(x_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Accuracy 
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            outputs = model(x_batch)

            preds = (outputs > 0.5).float()  # BCELoss → already sigmoid output

            y_batch = y_batch.view(-1)

            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)   

    accuracy = correct / total

    if (epoch + 1) % 5 == 0:
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | Acc: {accuracy:.2f}")

Epoch  5/40 | Loss: 0.5793 | Acc: 0.55
Epoch 10/40 | Loss: 0.1236 | Acc: 0.97
Epoch 15/40 | Loss: 0.0264 | Acc: 1.00
Epoch 20/40 | Loss: 0.0233 | Acc: 1.00
Epoch 25/40 | Loss: 0.0035 | Acc: 1.00
Epoch 30/40 | Loss: 0.0024 | Acc: 1.00
Epoch 35/40 | Loss: 0.0014 | Acc: 1.00
Epoch 40/40 | Loss: 0.0011 | Acc: 1.00


**Step 8 — Test With Your Own Sentences!**

In [8]:
def predict(text, model, word2idx):
    model.eval()

    encoded = encode_and_pad(text.lower(), word2idx)
    tensor = torch.tensor([encoded], dtype=torch.long)

    with torch.no_grad():
        outputs = model(tensor)
        prob = torch.sigmoid(outputs).item()


    if prob > 0.65:
        sentiment = "Positive 😊"
    elif prob < 0.35:
        sentiment = "Negative 😞"
    else:
        sentiment = "Neutral 😐"

    print(f"Text       : {text}")
    print(f"Sentiment  : {sentiment}")
    print(f"Confidence : {prob * 100:.2f}%")
    print("-" * 40)

    return sentiment, prob 

# ── Test it! ──────────────────────────────────
predict("I love this",          model, word2idx)
predict("this is terrible",     model, word2idx)
predict("amazing experience",   model, word2idx)
predict("I hate this",          model, word2idx)

# ── Test with  name! ───────────────────────
predict("Deepika loves learning AI", model, word2idx)

        

Text       : I love this
Sentiment  : Positive 😊
Confidence : 99.89%
----------------------------------------
Text       : this is terrible
Sentiment  : Negative 😞
Confidence : 0.08%
----------------------------------------
Text       : amazing experience
Sentiment  : Positive 😊
Confidence : 99.89%
----------------------------------------
Text       : I hate this
Sentiment  : Negative 😞
Confidence : 0.05%
----------------------------------------
Text       : Deepika loves learning AI
Sentiment  : Positive 😊
Confidence : 97.78%
----------------------------------------


('Positive 😊', 0.9777795076370239)